**Prérequis : `01_ingestion_bronze.ipynb`** exécuté (Bronze disponible dans `data/bronze/`).

---

# TP2 — Nettoyage et constitution de la couche Silver

**Cas d'usage :** prédiction du churn — éditeur SaaS B2B.

**Portée de ce notebook.** Partir de la couche **Bronze** (`data/bronze/`, produite par
`01_ingestion_bronze.ipynb`) — jamais des fichiers sources bruts directement, c'est
tout l'intérêt du Bronze : servir de source de vérité stable. Ce notebook corrige,
colonne par colonne, chaque anomalie recensée (et non corrigée) en Bronze, et produit
la couche **Silver** : une table propre, typée, dédoublonnée, jointe au référentiel
des plans.

**Ce qui n'est volontairement pas refait ici.** Le contrôle RGPD (motifs
nominatifs, colonnes identifiantes) a déjà été exécuté de façon systématique en amont
(`00_conformite_rgpd_anonymisation.ipynb`, gate GO) — inutile de le repasser.

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

BRONZE_DIR = Path("../data/bronze")
SILVER_DIR = Path("../data/silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_AT = datetime.now(timezone.utc).isoformat()
LAYER_VERSION = "v1"


def sha256_of(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

clients = pd.read_parquet(BRONZE_DIR / "clients_churn_bronze.parquet")
catalogue = pd.read_parquet(BRONZE_DIR / "catalogue_plans_bronze.parquet")

print(f"clients (Bronze) : {clients.shape[0]} lignes x {clients.shape[1]} colonnes")
print(f"catalogue (Bronze) : {catalogue.shape[0]} lignes x {catalogue.shape[1]} colonnes")

clients (Bronze) : 5035 lignes x 31 colonnes
catalogue (Bronze) : 4 lignes x 8 colonnes


## §1 — Détection et suppression des doublons

Doublons stricts sur les colonnes métier (les colonnes de traçabilité Bronze
`_source_file` / `_ingested_at_utc` sont exclues de la comparaison, sans quoi deux
lignes identiques ingérées à la même seconde ne seraient jamais détectées comme
doublons).

In [2]:
cols_metier = [c for c in clients.columns if not c.startswith("_")]
n_avant = len(clients)
n_doublons = clients.duplicated(subset=cols_metier).sum()

clients = clients.drop_duplicates(subset=cols_metier, keep="first").reset_index(drop=True)

print(f"Lignes avant   : {n_avant}")
print(f"Doublons retirés : {n_doublons}")
print(f"Lignes après   : {len(clients)}")

Lignes avant   : 5035
Doublons retirés : 35
Lignes après   : 5000


## §2 — Dates multi-formats

Trois formats coexistent dans `date_souscription` : ISO (`AAAA-MM-JJ`),
`JJ/MM/AAAA`, et texte anglais (`JJ Mon AAAA`, ex. `23 Dec 2024`).

In [3]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"]


def parse_date_multi(series: pd.Series, formats: list[str]) -> pd.Series:
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    still_missing = result.isna()
    for fmt in formats:
        parsed = pd.to_datetime(series.where(still_missing), format=fmt, errors="coerce")
        result = result.where(~parsed.notna(), parsed)
        still_missing = result.isna()
    return result


clients["date_souscription"] = parse_date_multi(clients["date_souscription"], DATE_FORMATS)
n_non_parsees = clients["date_souscription"].isna().sum()
print(f"Dates non parsées sur {len(clients)} lignes : {n_non_parsees}")
print("Étendue :", clients["date_souscription"].min(), "→", clients["date_souscription"].max())

Dates non parsées sur 5000 lignes : 0
Étendue : 2022-02-15 00:00:00 → 2025-01-10 00:00:00


## §3 — Nombres stockés en texte

Plusieurs colonnes numériques contiennent des virgules décimales, des symboles
(`%`, `€`) ou des unités (`h`) collés à la valeur. Un simple `pd.to_numeric` échoue
silencieusement sur ces valeurs et les traite comme manquantes — **un bug qui gonfle
artificiellement le taux de NaN**, illustré ci-dessous avant d'être corrigé.

In [4]:
NUMERIC_COLUMNS = [
    "anciennete_mois", "sieges_souscrits", "utilisateurs_actifs", "taux_adoption_pct",
    "connexions_30j", "heures_usage_30j", "fonctionnalites_total", "fonctionnalites_utilisees",
    "nb_integrations", "derniere_connexion_jours", "tickets_support_90j",
    "delai_reponse_support_h", "csat", "retards_paiement_12m",
    "revenu_mensuel_recurrent_eur", "sante_compte_fin_periode", "valeur_vie_client_eur", "churn",
]

# Piège : to_numeric brut échoue sur "33,3", "20.0%", "0.5 h", "280.62 €"...
col_demo = "revenu_mensuel_recurrent_eur"
naif = pd.to_numeric(clients[col_demo], errors="coerce")
vides_reels = (clients[col_demo].str.strip() == "").sum()
print(f"[{col_demo}] NaN avec to_numeric brut : {naif.isna().sum()} (dont {vides_reels} réellement vides)")
print("=> l'écart correspond aux valeurs mal formées ('40,0', '280.62 €'...), pas à de vraies données manquantes.")

[revenu_mensuel_recurrent_eur] NaN avec to_numeric brut : 1851 (dont 150 réellement vides)
=> l'écart correspond aux valeurs mal formées ('40,0', '280.62 €'...), pas à de vraies données manquantes.


In [5]:
def parse_numeric_fr(series: pd.Series) -> pd.Series:
    """Nettoie virgule décimale, %, € et unité 'h' avant conversion numérique."""
    s = series.astype(object).astype(str).str.strip()
    s = s.replace("", np.nan)
    s = s.str.replace("€", "", regex=False)
    s = s.str.replace("%", "", regex=False)
    s = s.str.replace("h", "", regex=False)
    s = s.str.strip()
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


rapport = []
for col in NUMERIC_COLUMNS:
    vides_reels = (clients[col].str.strip() == "").sum()
    fixed = parse_numeric_fr(clients[col])
    ecart = fixed.isna().sum() - vides_reels
    rapport.append({"colonne": col, "vides_reels": int(vides_reels), "nan_apres_fix": int(fixed.isna().sum()), "ecart": int(ecart)})
    clients[col] = fixed

rapport_df = pd.DataFrame(rapport)
print(rapport_df.to_string(index=False))
assert (rapport_df["ecart"] == 0).all(), "un écart résiduel signale un format encore mal géré"
print()
print("=> 0 écart résiduel : le taux de NaN par colonne correspond maintenant aux vraies valeurs manquantes.")

                     colonne  vides_reels  nan_apres_fix  ecart
             anciennete_mois            0              0      0
            sieges_souscrits            0              0      0
         utilisateurs_actifs            0              0      0
           taux_adoption_pct          250            250      0
              connexions_30j            0              0      0
            heures_usage_30j          300            300      0
       fonctionnalites_total            0              0      0
   fonctionnalites_utilisees            0              0      0
             nb_integrations          200            200      0
    derniere_connexion_jours            0              0      0
         tickets_support_90j            0              0      0
     delai_reponse_support_h          500            500      0
                        csat          400            400      0
        retards_paiement_12m          250            250      0
revenu_mensuel_recurrent_eur          15

## §4 — Catégorielles : casse, espaces et encodage

`plan` et `taille_entreprise` ne posent qu'un problème de casse/espaces (normalisable
directement). `secteur` porte en plus une corruption d'encodage sur les valeurs
accentuées (`Santé`, `Éducation`) — réparer l'encodage n'est pas fiable, on normalise
donc par **correspondance de sous-chaîne stable** (la partie ASCII du mot ne varie
pas, seul le caractère accentué est corrompu).

In [6]:
clients["plan"] = clients["plan"].str.strip().str.upper().str.capitalize()
clients["taille_entreprise"] = clients["taille_entreprise"].str.strip().str.upper()
clients["pays"] = clients["pays"].str.strip()

SECTEUR_MAP = [
    ("TECH", "Tech"),
    ("FINANC", "Finance"),
    ("COMMERC", "Commerce"),
    ("SANT", "Santé"),
    ("INDUSTR", "Industrie"),
    ("PUBLIC", "Public"),
    ("DUCATION", "Éducation"),
]


def normalize_secteur(value: str) -> str:
    v = (value or "").strip().upper()
    if v == "":
        return np.nan
    for needle, canon in SECTEUR_MAP:
        if needle in v:
            return canon
    return value  # valeur inattendue : laissée telle quelle, à investiguer


clients["secteur"] = clients["secteur"].map(normalize_secteur)

print("plan               :", sorted(clients["plan"].dropna().unique().tolist()))
print("taille_entreprise  :", sorted(clients["taille_entreprise"].dropna().unique().tolist()))
print("secteur            :", sorted(clients["secteur"].dropna().unique().tolist()))
print("pays               :", sorted(clients["pays"].unique().tolist()))

plan               : ['Business', 'Enterprise', 'Pro', 'Starter']
taille_entreprise  : ['ETI', 'GE', 'PME', 'TPE']
secteur            : ['Commerce', 'Finance', 'Industrie', 'Public', 'Santé', 'Tech', 'Éducation']
pays               : ['', 'Allemagne', 'Belgique', 'Canada', 'Espagne', 'France', 'Suisse']


## §5 — Jointure avec le catalogue des plans

Maintenant que `plan` est normalisé des deux côtés, la jointure ne doit plus laisser
d'orphelins.

In [7]:
catalogue["plan"] = catalogue["plan"].str.strip().str.capitalize()
catalogue_cols = ["plan", "prix_mensuel_par_siege_eur", "fonctionnalites_incluses",
                   "sla_reponse_h", "quota_stockage_go", "support_dedie"]
for c in ["prix_mensuel_par_siege_eur", "fonctionnalites_incluses", "sla_reponse_h", "quota_stockage_go"]:
    catalogue[c] = pd.to_numeric(catalogue[c], errors="coerce")

clients = clients.merge(catalogue[catalogue_cols], on="plan", how="left")

orphelins = clients["prix_mensuel_par_siege_eur"].isna().sum()
print(f"Lignes sans correspondance dans le catalogue après jointure : {orphelins}")
assert orphelins == 0, "la normalisation de 'plan' n'est pas complète"
print("Jointure réussie : 0 orphelin.")

Lignes sans correspondance dans le catalogue après jointure : 0
Jointure réussie : 0 orphelin.


## §6 — Valeurs manquantes : stratégie par colonne

Pas d'imputation uniforme — une décision par colonne, documentée.

In [8]:
strategie_nan = pd.DataFrame([
    {"colonne": "commentaire_csm", "taux_nan_pct": round(100 * clients["commentaire_csm"].eq("").mean(), 1),
     "strategie": "Laissée telle quelle", "justification": "Texte libre, pas une feature directe"},
    {"colonne": "revenu_mensuel_recurrent_eur", "taux_nan_pct": round(100 * clients["revenu_mensuel_recurrent_eur"].isna().mean(), 1),
     "strategie": "Recalcul via sièges × prix catalogue", "justification": "Plus fiable qu'une médiane globale"},
    {"colonne": "secteur / pays", "taux_nan_pct": round(100 * clients["secteur"].isna().mean(), 1),
     "strategie": "'Inconnu' explicite", "justification": "Catégorielle : pas de moyenne possible, doit rester traçable"},
    {"colonne": "numériques restants", "taux_nan_pct": None,
     "strategie": "Médiane documentée", "justification": "Robuste aux valeurs extrêmes déjà observées"},
])
strategie_nan

,colonne,taux_nan_pct,strategie,justification
0,commentaire_csm,55.5,Laissée telle quelle,"Texte libre, pas une feature directe"
1,revenu_mensuel_recurrent_eur,3.0,Recalcul via sièges × prix catalogue,Plus fiable qu'une médiane globale
2,secteur / pays,5.0,'Inconnu' explicite,"Catégorielle : pas de moyenne possible, doit r..."
3,numériques restants,NaN,Médiane documentée,Robuste aux valeurs extrêmes déjà observées


In [9]:
# 1) commentaire_csm : laissé tel quel (chaîne vide conservée, pas d'imputation)

# 2) revenu_mensuel_recurrent_eur : recalcul via sieges_souscrits x prix catalogue
mask_mrr_manquant = clients["revenu_mensuel_recurrent_eur"].isna()
n_recalc = mask_mrr_manquant.sum()
clients.loc[mask_mrr_manquant, "revenu_mensuel_recurrent_eur"] = (
    clients.loc[mask_mrr_manquant, "sieges_souscrits"] * clients.loc[mask_mrr_manquant, "prix_mensuel_par_siege_eur"]
)
print(f"MRR recalculé pour {n_recalc} lignes ({100 * n_recalc / len(clients):.1f}%)")

# 3) secteur / pays : "Inconnu" explicite
for col in ["secteur", "pays"]:
    n_manquant = clients[col].isna().sum() if col == "secteur" else (clients[col].str.strip() == "").sum()
    if col == "pays":
        clients[col] = clients[col].replace("", "Inconnu")
    else:
        clients[col] = clients[col].fillna("Inconnu")
    print(f"{col} : {n_manquant} valeurs -> 'Inconnu'")

# 4) numériques restants : médiane documentée
colonnes_mediane = [
    "taux_adoption_pct", "heures_usage_30j", "delai_reponse_support_h",
    "csat", "retards_paiement_12m", "nb_integrations",
]
for col in colonnes_mediane:
    n_manquant = clients[col].isna().sum()
    mediane = clients[col].median()
    clients[col] = clients[col].fillna(mediane)
    print(f"{col} : {n_manquant} valeurs -> médiane ({mediane:.1f})")

MRR recalculé pour 150 lignes (3.0%)
secteur : 250 valeurs -> 'Inconnu'
pays : 200 valeurs -> 'Inconnu'
taux_adoption_pct : 250 valeurs -> médiane (50.0)
heures_usage_30j : 300 valeurs -> médiane (5.7)
delai_reponse_support_h : 500 valeurs -> médiane (10.5)
csat : 400 valeurs -> médiane (3.0)
retards_paiement_12m : 250 valeurs -> médiane (0.0)
nb_integrations : 200 valeurs -> médiane (2.0)


## §7 — Persistance de la couche Silver

**Note d'environnement.** TP2 ciblait à l'origine un chargement dans PostgreSQL
(`churn_saas_db.clients_churn`) — cible de production toujours valide. Cette machine
ne dispose pas d'un serveur PostgreSQL accessible depuis ce notebook : la couche
Silver est donc persistée en Parquet dans `data/silver/`, cohérent avec la couche
Bronze qui suit le même principe.

In [10]:
silver_path = SILVER_DIR / "clients_churn_silver.parquet"
clients_out = clients.drop(columns=["_source_file", "_ingested_at_utc"])
clients_out.to_parquet(silver_path, index=False)

manifest = {
    "couche": "silver",
    "version": LAYER_VERSION,
    "processed_at_utc": PROCESSED_AT,
    "source": "data/bronze/clients_churn_bronze.parquet + catalogue_plans_bronze.parquet",
    "sha256_source_bronze": {
        "clients_churn_bronze.parquet": sha256_of(BRONZE_DIR / "clients_churn_bronze.parquet"),
        "catalogue_plans_bronze.parquet": sha256_of(BRONZE_DIR / "catalogue_plans_bronze.parquet"),
    },
    "lignes": int(len(clients_out)),
    "colonnes": list(clients_out.columns),
    "transformations": [
        f"{n_doublons} doublons stricts supprimés",
        "dates converties (3 formats -> datetime64)",
        "nombres texte convertis (virgule, %, €, unité 'h' nettoyés)",
        "casse/espaces normalisés (plan, taille_entreprise) ; secteur normalisé par sous-chaîne stable",
        "jointure avec catalogue_plans (0 orphelin)",
        "valeurs manquantes traitées colonne par colonne (voir §6)",
    ],
    "cible_production": "PostgreSQL churn_saas_db.clients_churn (non disponible dans cet environnement)",
    "chemin_silver": str(silver_path),
    "sha256_silver": sha256_of(silver_path),
}
manifest_path = SILVER_DIR / "silver_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Couche Silver écrite :", silver_path)
print("Manifeste écrit      :", manifest_path)

Couche Silver écrite : ..\data\silver\clients_churn_silver.parquet
Manifeste écrit      : ..\data\silver\silver_manifest.json


## §8 — Vérification finale

In [11]:
verif = pd.DataFrame([
    {"vérification": "Lignes", "résultat": len(clients_out)},
    {"vérification": "Colonnes", "résultat": clients_out.shape[1]},
    {"vérification": "Doublons résiduels", "résultat": int(clients_out.duplicated().sum())},
    {"vérification": "NaN résiduels (hors commentaire_csm)",
     "résultat": int(clients_out.drop(columns=["commentaire_csm"]).isna().sum().sum())},
    {"vérification": "Lignes sans correspondance catalogue",
     "résultat": int(clients_out["prix_mensuel_par_siege_eur"].isna().sum())},
])
verif

,vérification,résultat
0,Lignes,5000
1,Colonnes,34
2,Doublons résiduels,0
3,NaN résiduels (hors commentaire_csm),0
4,Lignes sans correspondance catalogue,0


## Journal de bord — Synthèse TP2 (Silver)

- **Doublons.** Détectés et supprimés sur les colonnes métier (traçabilité Bronze
  exclue du calcul).
- **Dates.** 3 formats unifiés en `datetime64`, 0 date non parsée.
- **Nombres en texte.** Bug démontré (to_numeric brut gonfle le NaN sur les valeurs
  avec virgule/%/€/unité), corrigé par un parseur dédié — 0 écart résiduel sur toutes
  les colonnes numériques.
- **Catégorielles.** Casse/espaces normalisés (plan, taille_entreprise) ; secteur
  normalisé par correspondance de sous-chaîne stable (l'encodage corrompu n'est pas
  réparé, contourné).
- **Jointure.** 0 orphelin avec le catalogue des plans après normalisation.
- **Valeurs manquantes.** Stratégie par colonne, pas d'imputation uniforme — MRR
  recalculé plutôt qu'imputé.
- **RGPD.** Non retraité ici — déjà couvert en amont par le portique dédié.
- **Persistance.** `data/silver/clients_churn_silver.parquet` (+ manifeste), cible de
  production PostgreSQL documentée mais non disponible dans cet environnement.
- **Prochaine étape.** Notebook Gold — anti-fuite, feature engineering, split
  train/test.